# 🎵 Análisis Morfosintáctico de Letras Musicales

**Pipeline completo:** Scraping → Limpieza → Almacenamiento en MongoDB

| Módulo | Archivo | Responsabilidad |
|---|---|---|
| Scraper | `src/data/scraper_api.py` | Consulta API Genius + scraping de letras |
| Limpieza | `src/processing/lyrics_cleaner.py` | Limpieza estructural y para NLP |
| Almacenamiento | `src/storage/mongo_storage.py` | Persistencia y consultas MongoDB |

## 0. Configuración del entorno

In [1]:
import sys
from pathlib import Path
import os
import pandas as pd
sys.path.append(os.path.abspath(".."))
from src.data import limpiar_letra
from src.data.mongo_storage import (
    insertar_canciones,
    listar_artistas,
    get_collection,
)
from src.scraper.scraper_api import extraer_multiples_artistas


## 1. Parámetros globales

> ⚠️ Cambia `GENIUS_TOKEN` por tu token personal de [genius.com/api-clients](https://genius.com/api-clients)

In [2]:
# ── Credenciales ──────────────────────────────────────────────────────────────
GENIUS_TOKEN = "3HbLMT-naLmQ87FG4xKJyJgVZXA034rf8DYQp3oxBjlq74Kq2Cq4t8Rd7lntDilM"   # <-- reemplaza con tu token

# ── MongoDB ───────────────────────────────────────────────────────────────────
MONGO_URI = "mongodb://localhost:27017"
MONGO_DB  = "musica"
MONGO_COL = "canciones"

# ── Scraping ──────────────────────────────────────────────────────────────────
CANCIONES_POR_ARTISTA = 10
# Subir un nivel (..) y entrar a data/raw/archivo.csv
ruta_csv = os.path.join("..", "data", "raw", "artistas.csv")

df_artistas = pd.read_csv(ruta_csv)

print(f'Configuración lista — {CANCIONES_POR_ARTISTA} canciones por artista')

Configuración lista — 10 canciones por artista


## 2. Cargar lista de artistas desde CSV

In [3]:
# ✅ Celda 6 corregida — mantiene artist + genre
col_artist = [c for c in df_artistas.columns if c.lower() in ('artist', 'artista')][0]
col_genre  = [c for c in df_artistas.columns if c.lower() in ('genre', 'género', 'genero')][0]

lista_artistas = (
    df_artistas[[col_artist, col_genre]]
    .dropna()
    .rename(columns={col_artist: "artist", col_genre: "genre"})
    .to_dict(orient="records")
)

print(f'{len(lista_artistas)} artistas cargados:')
for a in lista_artistas:
    print(f'   • {a["artist"]} [{a["genre"]}]')

140 artistas cargados:
   • Taylor Swift [Pop]
   • Olivia Rodrigo [Pop]
   • Billie Eilish [Pop]
   • Shawn Mendes [Pop]
   • Harry Styles [Pop]
   • Katy Perry [Pop]
   • Lady Gaga [Pop]
   • Justin Bieber [Pop]
   • Sabrina Carpenter [Pop]
   • Troye Sivan [Pop]
   • Zara Larsson [Pop]
   • Ava Max [Pop]
   • Tate McRae [Pop]
   • Conan Gray [Pop]
   • Raye [Pop]
   • Madison Beer [Pop]
   • Lorde [Pop]
   • Camila Cabello [Pop]
   • Niall Horan [Pop]
   • Halsey [Pop]
   • Kendrick Lamar [Hip-Hop]
   • J. Cole [Hip-Hop]
   • Travis Scott [Hip-Hop]
   • Lil Baby [Hip-Hop]
   • 21 Savage [Hip-Hop]
   • Tyler, The Creator [Hip-Hop]
   • Doja Cat [Hip-Hop]
   • Megan Thee Stallion [Hip-Hop]
   • Cardi B [Hip-Hop]
   • A$AP Rocky [Hip-Hop]
   • Juice WRLD [Hip-Hop]
   • Pop Smoke [Hip-Hop]
   • Roddy Ricch [Hip-Hop]
   • Logic [Hip-Hop]
   • Polo G [Hip-Hop]
   • NF [Hip-Hop]
   • Jack Harlow [Hip-Hop]
   • Lil Uzi Vert [Hip-Hop]
   • Yeat [Hip-Hop]
   • Central Cee [Hip-Hop]
   • Luke 

## 3. Scraping de letras

In [4]:
canciones = extraer_multiples_artistas(
    lista_artistas=lista_artistas,
    token=GENIUS_TOKEN,
    cantidad=CANCIONES_POR_ARTISTA,
)

print(f'\n🎵 Total canciones extraídas: {len(canciones)}')


──────────────────────────────────────────────────
 Extrayendo 10 canciones de: Taylor Swift [Pop]
──────────────────────────────────────────────────
  → All Too Well (10 Minute Version) (Taylor’s Version) [From The Vault]
  → Wood
  → All Too Well (10 Minute Version) (Taylor’s Version) [Live Acoustic]
  → Fortnight
  → The Fate of Ophelia
  → cardigan
  → loml
  → So Long, London
  → The Tortured Poets Department
  → deja vu
  ✓ 10 canciones obtenidas de Taylor Swift


──────────────────────────────────────────────────
 Extrayendo 10 canciones de: Olivia Rodrigo [Pop]
──────────────────────────────────────────────────
  → drivers license
  → deja vu
  → good 4 u
  → vampire
  → All I Want
  → happier
  → traitor
  → lacy
  → the grudge
  → get him back!
  ✓ 10 canciones obtenidas de Olivia Rodrigo


──────────────────────────────────────────────────
 Extrayendo 10 canciones de: Billie Eilish [Pop]
──────────────────────────────────────────────────
  → lovely
  → WILDFLOWER
  → when t

In [5]:
# Vista previa de las canciones obtenidas
df_preview = pd.DataFrame(canciones)
df_preview[['Song', 'Artist', 'Genre', 'Song year', 'Language']].head(15)

,Song,Artist,Genre,Song year,Language
0,All Too Well (10 Minute Version) (Taylor’s Ver...,Taylor Swift,Pop,2021,en
1,Wood,Taylor Swift,Pop,2025,en
2,All Too Well (10 Minute Version) (Taylor’s Ver...,Taylor Swift,Pop,2021,en
3,Fortnight,Taylor Swift,Pop,2024,en
4,The Fate of Ophelia,Taylor Swift,Pop,2025,en
5,cardigan,Taylor Swift,Pop,2020,en
6,loml,Taylor Swift,Pop,2024,en
7,"So Long, London",Taylor Swift,Pop,2024,en
8,The Tortured Poets Department,Taylor Swift,Pop,2024,en
9,deja vu,Taylor Swift,Pop,2021,en


## 4. Guardar en MongoDB

In [6]:
collection = get_collection(uri=MONGO_URI, db=MONGO_DB, col=MONGO_COL)

resumen = insertar_canciones(canciones, collection=collection)

print('Resultado del upsert en MongoDB:')
print(f'   • Nuevas insertadas : {resumen["insertados"]}')
print(f'   • Ya existían       : {resumen["actualizados"]}')

Resultado del upsert en MongoDB:
   • Nuevas insertadas : 1311
   • Ya existían       : 83
